# 3b — Export Coding Materials

Builds human coding materials for the Stage 3 validation sample.

**Inputs:**
- `data/chunk_registry_v2.parquet` or CSV fallback

**Outputs:**
- `data/human_coding/materials/A|B|C/*.csv`
- `data/human_coding/materials/utterance/*.csv`
- `data/human_coding/videos/*.mp4` (when source videos are available)
- `data/human_coding/materials/validation_chunk_manifest.csv`


## Step 0 — Imports and paths

In [ ]:
import shutil
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm

NOTEBOOK_DIR = Path().resolve()
ANALYSIS_V2 = NOTEBOOK_DIR.parent
DATA_DIR = ANALYSIS_V2 / 'data'
HUMAN_CODING_DIR = DATA_DIR / 'human_coding'
MATERIALS_DIR = HUMAN_CODING_DIR / 'materials'
VIDEOS_DIR = HUMAN_CODING_DIR / 'videos'

for path in [MATERIALS_DIR / 'A', MATERIALS_DIR / 'B', MATERIALS_DIR / 'C', MATERIALS_DIR / 'utterance', VIDEOS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print('HUMAN_CODING_DIR:', HUMAN_CODING_DIR)


## Step 1 — Load validation registry

In [ ]:
def load_table(stem: str) -> pd.DataFrame:
    parquet_path = DATA_DIR / f'{stem}.parquet'
    csv_path = DATA_DIR / f'{stem}.csv'
    if parquet_path.exists():
        try:
            return pd.read_parquet(parquet_path)
        except Exception as e:
            print(f'Falling back to CSV for {stem}: {type(e).__name__}: {e}')
    if csv_path.exists():
        return pd.read_csv(csv_path)
    raise FileNotFoundError(f'Could not find {parquet_path.name} or {csv_path.name}')


registry = load_table('chunk_registry_v2')
val_registry = registry.loc[registry['human_validation_set'] == True].copy()
utterance_registry = registry.loc[registry['utterance_validation_set'] == True].copy()

print(f'Chunk-level validation chunks: {len(val_registry):,}')
print(f'Utterance-level validation chunks: {len(utterance_registry):,}')
val_registry.head(2)


## Step 2 — Define instruments and export coding sheets

In [ ]:
INSTRUMENT_A_FIELDS = [
    'idea_trajectory',
    'idea_trajectory_justification',
    'problem_specificity_level',
    'problem_specificity_justification',
    'decision_crystallization_level',
    'decision_crystallization_justification',
    'ambition_level',
    'cross_disciplinary_bridging',
    'cross_disciplinary_bridging_description',
    'explicit_commitment_signal',
    'commitment_signal_quote',
]
INSTRUMENT_B_FIELDS = [
    'pronoun_shift_flag',
    'shared_vision_indicator',
    'shared_vision_quote',
    'laughter_quality',
    'personal_disclosure',
    'dissent_response_quality',
    'risk_acknowledgment_with_enthusiasm',
    'risk_enthusiasm_quote',
    'meeting_structure_quality',
]
INSTRUMENT_C_FIELDS = [
    'collective_engagement_level',
    'collective_engagement_justification',
]
UTTERANCE_FIELDS = [
    'utterance_index',
    'speaker',
    'start_time',
    'end_time',
    'transcript',
    'Idea_Management_subcode',
    'Integration_Practices_subcode',
    'Pronoun_Framing_subcode',
    'interruption_type',
    'notes',
]


def export_chunk_sheet(chunk_id: str, fields: list[str], out_dir: Path, instrument_label: str):
    rows = [{'field': f, 'rater_code': '', 'notes': ''} for f in fields]
    pd.DataFrame(rows).to_csv(out_dir / f'{chunk_id}__instrument_{instrument_label}.csv', index=False)


for _, row in tqdm(val_registry.iterrows(), total=len(val_registry), desc='Exporting chunk sheets', unit='chunk'):
    chunk_id = row['chunk_id']
    export_chunk_sheet(chunk_id, INSTRUMENT_A_FIELDS, MATERIALS_DIR / 'A', 'A')
    export_chunk_sheet(chunk_id, INSTRUMENT_B_FIELDS, MATERIALS_DIR / 'B', 'B')
    export_chunk_sheet(chunk_id, INSTRUMENT_C_FIELDS, MATERIALS_DIR / 'C', 'C')

for _, row in tqdm(utterance_registry.iterrows(), total=len(utterance_registry), desc='Exporting utterance sheets', unit='chunk'):
    chunk_id = row['chunk_id']
    pd.DataFrame(columns=UTTERANCE_FIELDS).to_csv(MATERIALS_DIR / 'utterance' / f'{chunk_id}__utterance_validation.csv', index=False)

print('Coding sheets exported.')


## Step 3 — Copy chunk videos when available

In [ ]:
copied = []
missing = []

for _, row in tqdm(val_registry.iterrows(), total=len(val_registry), desc='Copying videos', unit='chunk'):
    src = Path(str(row['chunk_path']))
    if src.exists():
        dst = VIDEOS_DIR / src.name
        shutil.copy2(src, dst)
        copied.append(dst.name)
    else:
        missing.append({'chunk_id': row['chunk_id'], 'chunk_path': str(src)})

print(f'Videos copied: {len(copied):,}')
print(f'Videos missing: {len(missing):,}')
pd.DataFrame(missing).head(10)


## Step 4 — Save manifest

In [ ]:
manifest = val_registry[[
    'chunk_id', 'conference_id', 'session_group', 'chunk_position', 'chunk_path',
    'human_validation_set', 'utterance_validation_set', 'oversampled_for'
]].copy()

manifest_path = MATERIALS_DIR / 'validation_chunk_manifest.csv'
manifest.to_csv(manifest_path, index=False)
print('Saved:', manifest_path)
manifest.head(10)
